# 03 — TCP Models (OS proxy + RANO v3)

- Pooled TCP curves (OS ≥ median)
- RANO non-PD vs OS on same patients
- Within-arm DVH → RANO (60 / 40 Gy)
- Cox OS ~ EQD2 + RANO


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import DATA_PROCESSED, FIGURES_DIR, REPORTS_DIR
from src.models.poisson_tcp import PoissonTCPModel
from src.models.logistic_tcp import LogisticTCPModel
from src.models.probit_tcp import ProbitTCPModel
from src.models.model_comparison import run_model_comparison
from src.analysis.rano_tcp_comparison import run_rano_tcp_comparison
from src.analysis.within_arm_rano_tcp import run_within_arm_rano_analysis


In [ ]:
modeling = pd.read_csv(DATA_PROCESSED / "modeling_table.csv")
median_os = modeling["survival_weeks"].median()
outcomes = (modeling["survival_weeks"] >= median_os).astype(float).to_numpy()
doses = modeling["eqd2_gy"].to_numpy()
print(f"n={len(modeling)}, OS median={median_os:.0f} wk, RANO labels={modeling['rano_controlled_t1'].notna().sum()}")


In [ ]:
models = [
    ("Poisson", PoissonTCPModel(d50_init=55, gamma50_init=1.5)),
    ("Logistic", LogisticTCPModel(d50_init=53, k_init=0.1)),
    ("Probit", ProbitTCPModel(d50_init=53, sigma_init=10)),
]
d_grid = np.linspace(35, 65, 200)
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, m in models:
    m.fit(doses, outcomes)
    ax.plot(d_grid, m.predict(d_grid), label=name, lw=2)
ax.set_xlabel("EQD2 (Gy)")
ax.set_ylabel("TCP (OS ≥ median proxy)")
ax.set_title(f"Pooled TCP (n={len(modeling)})")
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "03_tcp_curves_os_proxy.png")
plt.close(fig)

comparison_df, _ = run_model_comparison(doses, outcomes, frame=modeling)
comparison_df[["model", "aic", "roc_auc", "hl_p_value"]]


In [ ]:
metrics_dir = REPORTS_DIR / "metrics"
rano_cmp, _, _ = run_rano_tcp_comparison(modeling, metrics_dir)
rano_cmp[["model", "endpoint", "roc_auc_insample", "roc_auc_cv_mean", "lr_p_value"]]


In [ ]:
within_df, cox_rano = run_within_arm_rano_analysis(modeling, metrics_dir)
within_df[["scheme", "metric", "n_rano", "spearman_rho", "spearman_p", "poisson_auc", "poisson_lr_p"]]


In [ ]:
cox_rano[["term", "hazard_ratio", "p_value"]]
